# Uitwerkingen bij het demo-college — EDA op de toernooidata

Voor de docent. Per cel: wat je laat zien, wat de valkuil is, en het antwoord
op de vraag die erbij hoort. De getallen hieronder komen uit week 3 ronde 3
(27 bots, 6750 logregels) — na een nieuwe ronde schuiven ze, het verhaal niet.

Draai eerst de cellen van `03_demo_college_toernooi.ipynb`, dan werkt alles hier.

## De `winst`-kolom · waarom `diff()` en waarom `fillna`

Het log bewaart de stack **ná** elke hand. Wat een hand opleverde is dus het
verschil met de hand ervoor:

```python
handen.groupby(["bot_naam", "simulatie"])["stack"].diff()
```

Twee dingen die studenten hier fout doen:

1. **Groeperen zonder `simulatie`.** Simulaties zijn parallelle werelden: dezelfde
   bot speelt vijf keer een compleet toernooi. Zonder die groepering trekt `diff()`
   de eerste hand van simulatie 1 af van de laatste hand van simulatie 0 — een
   sprong van honderden chips die nooit is gespeeld.
2. **De `NaN` laten staan.** De eerste hand van elke zitting heeft geen voorganger.
   Die meet je vanaf de startstack: `.fillna(handen["stack"] - 1000)`.

Laat je punt 2 weg, dan mist er per bot vijf handen winst, en dan klopt de
nulsom-controle niet meer. Dat is precies het nut van die controle.

In [ ]:
# De reconstructie klopt exact -- dit is de dia waarmee je de rest verdient
per_bot = handen.groupby("bot_naam")["winst"].sum() / handen["simulatie"].nunique() + 1000
eindstand = pd.Series(resultaat["eindstand_per_bot"])
print("grootste afwijking:", (per_bot - eindstand).abs().max())
print("som van alle winst:", handen["winst"].sum())

## Filteren · de tegenhanger van `continent == 'Europe'`

Niets nieuws, maar wel het moment om te laten zien dat `handen[...]` een **kopie-achtige
selectie** teruggeeft en niet de tabel zelf. Zet je daar een kolom in met
`mijn_handen["iets"] = ...`, dan krijg je de `SettingWithCopyWarning`.

Antwoord op "hoeveel rijen zijn dat?": 250 per bot — 50 handen × 5 simulaties.

## `groupby(...)['winst'].sum()` · en de valkuil ernaast

De eerste cel is goed: winst is een **stroom**, die mag je optellen.

De tweede cel is fout, en dat is de hele les:

```python
handen.groupby("bot_naam")["stack"].sum()   # 258.550 chips?!
```

Een stack is een **stand**, geen stroom. Die optellen over 250 handen is hetzelfde
als de bevolking van Europa over alle gapminder-jaren bij elkaar optellen: je krijgt
een groot getal dat niets betekent. In gapminder kwam er 12 × de bevolking uit, hier
250 × de stack.

**De vraag aan de zaal:** het getal is netjes, de code geeft geen foutmelding, en het
staat in je dashboard. Hoe had je dit gemerkt? Antwoord: door te weten wat de eenheid
van je kolom is. Een som van chips-op-een-moment bestaat niet.

Voor een stand wil je `.last()` (waar stond hij aan het eind) of `.mean()`.

In [ ]:
# goed: stroom optellen, stand aflezen
print(handen.groupby("bot_naam")["winst"].sum().head(3))
print()
print(handen.groupby("bot_naam")["stack"].last().head(3))

## Twee niveaus · en waarom `reset_index()`

`groupby(["bot_naam", "hand_nummer"])` geeft een **MultiIndex**: bot en handnummer
zitten allebei in de index, niet in een kolom. Seaborn, Plotly en Streamlit willen
kolommen. Vandaar `.reset_index()`.

Laat het verschil één keer zien — dat scheelt later een half werkcollege:

In [ ]:
zonder = handen.groupby(["bot_naam", "hand_nummer"])["stack"].mean()
met = zonder.reset_index()
print(type(zonder), zonder.index.names)
print(list(met.columns))

**De brug naar Case 2.** Dit verloop per handnummer is precies wat het criterium
*Vergelijken en annoteren* vraagt: niet alleen de waarde, maar het verschil met het
moment ervoor (`diff`), het opgetelde verloop (`cumsum`) of de stand ten opzichte van
het begin (`shift`). Je hebt `diff` hierboven al gebruikt om `winst` te maken.

Zeg dat hardop. Studenten leggen die koppeling zelf niet.

In [ ]:
# het verloop van de koploper, met de drie werkwoorden uit de rubric
top = handen.groupby("bot_naam")["stack"].last().idxmax()
een = handen[(handen["bot_naam"] == top) & (handen["simulatie"] == 0)].copy()
een["verschil"] = een["stack"].diff()
een["opgeteld"] = een["winst"].cumsum()
een[["hand_nummer", "stack", "verschil", "opgeteld"]].head(8)

## De widget

Zelfde code als bij gapminder, andere kolommen: `stack`, `tafel`, `simulatie`,
`hand_nummer`, `winst`.

Kies **`winst`** als je er één laat zien. Je krijgt een enorme piek op 0 met twee
dunne staarten — en dat beeld is het argument voor het hele blok statistiek:
het gemiddelde is 0, de mediaan is 0, en toch zit er een hand van +1750 in.

`tafel` en `simulatie` zijn de anti-voorbeelden: technisch numeriek, inhoudelijk
labels. Een histogram daarvan zegt niets. Goede vraag aan de zaal: *welke van deze
vijf kolommen zijn écht getallen?*

## `apply()` met een lambda

```python
handen.groupby("bot_naam")["actie"].apply(lambda s: f"{(s == 'fold').mean():.0%} gefold")
```

Let op dat we `["actie"]` **vóór** de `apply` selecteren. Doe je de apply op de hele
groep (`groupby("bot_naam").apply(...)`), dan zit de groepskolom er in nieuwere pandas
nog bij en krijg je een `DeprecationWarning` over `include_groups`. Dat kost je in een
college vijf minuten die je niet hebt.

En het getal dat eruit komt is fout. Zie de volgende cel.

## "Wat doet dit stuk code?" · twee percentages die allebei kloppen lijken

```python
(handen["actie"] == "fold").mean()                      # 0,795
handen["actie"].value_counts(normalize=True)["fold"]    # 0,922
```

**79,5% tegen 92,2%, en geen van beide geeft een waarschuwing.**

De oorzaak zit in de 928 logregels (13,7%) waar `actie` op `None` staat: daar was de
bot helemaal niet aan zet — hij zat in de blinds en de hand was al afgelopen voordat
het zijn beurt was.

- `(kolom == "fold")` maakt van `None` een `False`. Die regels tellen dus mee in de
  noemer als "niet gefold". → te laag.
- `value_counts()` laat lege waarden standaard **vallen** (`dropna=True`). Die regels
  tellen helemaal niet mee. → dit is het percentage van de beslismomenten, en dat is
  wat je wilde weten.

Allebei gedragen zich redelijk; ze beantwoorden alleen een andere vraag. En je ziet
aan de code niet welke.

**Zo maak je het expliciet** — dan hoef je niemands standaardinstelling te kennen:

```python
handen.loc[handen["aan_zet"], "actie"]
```

`handen["aan_zet"]` is een kolom van `True`/`False`; die in `.loc[...]` zetten houdt
alleen de rijen waar hij `True` is. Dat geeft 92,2%, net als `value_counts` — maar nu
omdat jij het hebt opgeschreven.

**Dit is de belangrijkste cel van het college.** Niet omdat `.loc` moeilijk is, maar
omdat de `apply`-cel hiervóór in dezelfde val trapt: die gebruikt `.mean()` en geeft
voor bot 500851324 **84% gefold**, terwijl het over zijn echte beslissingen **96%**
is. Het verschil tussen "speelt voorzichtig" en "speelt niet".

Draai de cel hieronder om alle vier de getallen naast elkaar te zetten.

In [ ]:
print("logregels zonder actie:", handen["actie"].isna().sum(), "van", len(handen))
print()
print("(actie == fold).mean()          ", round((handen["actie"] == "fold").mean() * 100, 1))
print("value_counts(normalize=True)    ", round(handen["actie"].value_counts(normalize=True)["fold"] * 100, 1))
print("expliciet op aan_zet            ",
      round((handen.loc[handen["aan_zet"], "actie"] == "fold").mean() * 100, 1))
print("value_counts(..., dropna=False) ",
      round(handen["actie"].value_counts(normalize=True, dropna=False)["fold"] * 100, 1))

---

# Uitwerking van de drie TODO's

In [ ]:
# TODO 1 -- de eindstand als tabel
eind = pd.DataFrame(resultaat["eindstand_per_bot"].items(),
                    columns=["bot_naam", "eindstand"])
eind.sort_values("eindstand", ascending=False).head(6)

In [ ]:
# TODO 2 -- fold-percentage over alleen de beslismomenten, en dan de merge
fold = (handen[handen["aan_zet"]]
        .groupby("bot_naam")["actie"]
        .apply(lambda s: (s == "fold").mean() * 100)
        .rename("fold_pct")
        .reset_index())

print(len(eind), "->", end=" ")
samen = eind.merge(fold, on="bot_naam", how="inner")
print(len(samen))

samen.sort_values("eindstand", ascending=False).head(6)

`how="inner"` mag hier omdat beide tabellen uit hetzelfde toernooi komen: 27 in,
27 uit. Dat is geen aanname maar een controle — vandaar de twee `len()`s.

Wél opletten bij `vergelijk_met_week=1`: dan heten de bots `500959250__w3` en past
de sleutel niet meer op een tabel met kale studentnummers. Dat is dezelfde fout als
"Den Haag" tegenover "'s-Gravenhage" uit de Case 2-waarschuwing.

In [ ]:
# TODO 3 -- hangen ze samen?
samen["fold_pct"].corr(samen["eindstand"]).round(2)

**−0,69.** Negatief: hoe meer je foldt, hoe lager je eindigt.

Wat je er níét uit mag lezen: dat minder folden je vanzelf hoger zet. Vier bots
bepalen dit getal grotendeels, en met 27 punten is een correlatie een indruk, geen
bewijs. Dat onderscheid — *ik zie een verband* tegenover *ik heb iets aangetoond* —
is wat het criterium **Analyse** van Case 2 beoordeelt.

En dan de dia waarop je eindigt: bovenaan staat `Testbot_CalltAlles`, die 0% foldt,
niet naar zijn kaarten kijkt en één regel code is. Hij won met tien chips verschil.
De vraag voor donderdag is niet "moet ik minder folden" maar **"waar liggen mijn
drempels, en waarom daar?"**